In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import numpy as np

# [Core Modification 1]: Completely abandon the MONAI architecture and directly introduce the official underlying backbone library of nnU-Net v2
from dynamic_network_architectures.architectures.unet import PlainConvUNet
from dynamic_network_architectures.initialization.weight_init import InitWeights_He

# Data flow processing still uses MONAI, which does not affect the purity of the network architecture
from monai.transforms import (
    Compose, LoadImaged, SpatialPadd, RandSpatialCropd, RandZoomd, ToTensord
)
from monai.data import DataLoader, Dataset

# ==========================================
# 1. Pure nnU-Net v2 Architecture MIM Wrapper
# ==========================================
class MultiModal_nnUNetV2_MIM(nn.Module):
    def __init__(self, in_channels=2, patch_size=(12, 16, 16), mask_ratio=0.5):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.patch_size = patch_size
        
        # ------------------------------------------------------------------
        # [Core Modification 2]: Precisely replicate the heuristic structural parameters of nnU-Net v2 3d_fullres
        # For input size (96, 128, 128):
        # Stage 0: No downsampling (stride 1)
        # Stage 1-5: Downsample by factor of 2 (stride 2) each time, 5 times in total, reaching the bottleneck layer (3, 4, 4)
        # ------------------------------------------------------------------
        conv_kernel_sizes = [[3, 3, 3]] * 6
        strides = [[1, 1, 1], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2]]
        features_per_stage = [32, 64, 128, 256, 320, 320] # Default upper limit for medical 3D is 320
        n_conv_per_stage = [2, 2, 2, 2, 2, 2]
        n_conv_per_stage_decoder = [2, 2, 2, 2, 2]
        
        # Instantiate the true nnU-Net v2 core network
        self.nnunet = PlainConvUNet(
            input_channels=in_channels,
            n_stages=6,
            features_per_stage=features_per_stage,
            conv_op=nn.Conv3d,
            kernel_sizes=conv_kernel_sizes,
            strides=strides,
            n_conv_per_stage=n_conv_per_stage,
            num_classes=in_channels, # Reconstruction task: output channels = input channels
            n_conv_per_stage_decoder=n_conv_per_stage_decoder,
            conv_bias=True,
            norm_op=nn.InstanceNorm3d,
            norm_op_kwargs={'eps': 1e-5, 'affine': True},
            dropout_op=None, # No Dropout during pre-training
            nonlin=nn.LeakyReLU,
            nonlin_kwargs={'negative_slope': 1e-2, 'inplace': True},
            deep_supervision=False # Pre-training reconstruction directly outputs the final layer
        )
        
        # Signature He initialization of nnU-Net v2
        self.nnunet.apply(InitWeights_He(1e-2))

    def forward(self, x):
        B, C, Z, Y, X = x.shape
        pz, py, px = self.patch_size
        
        grid_z, grid_y, grid_x = Z // pz, Y // py, X // px
        
        # Generate Mask independently
        noise = torch.rand(B, C, grid_z, grid_y, grid_x, device=x.device)
        mask = (noise < self.mask_ratio).float() 
        
        mask_expanded = mask.repeat_interleave(pz, dim=2).repeat_interleave(py, dim=3).repeat_interleave(px, dim=4)
        
        # Mean Imputation (Set the masked areas to 0.0)
        x_masked = x * (1 - mask_expanded)
        
        # Feed into the pure nnU-Net v2 for dense feature extraction and reconstruction
        x_rec = self.nnunet(x_masked)
        
        # Weighted Global MSE Loss
        mse_loss = F.mse_loss(x_rec, x, reduction='none')
        loss_masked = (mse_loss * mask_expanded).sum() / (mask_expanded.sum() + 1e-8)
        loss_unmasked = (mse_loss * (1 - mask_expanded)).sum() / ((1 - mask_expanded).sum() + 1e-8)
        
        loss = loss_masked + 0.2 * loss_unmasked
        
        return loss, x_rec

# ==========================================
# 2. Dataset & Transforms (keeping your settings)
# ==========================================
def get_pretraining_dataloader(batch_size=16):
    base_path = Path('/path/to/your/dataset/PETCTfoundation/')
    anomalies = [
        'LDca4f40/LDca5687', 'LDca56d5/LDca5e13', 'LDca4eed/LDca54ed',
        'LDca519f/LDca5602', 'LDca56da/LDca5d8b', 'LDca58c7/LDca5c77',
        'LDca58c3/LDca5c6f', 'LDca4f3f/LDca5507', 'LDca58c2/LDca5c6e',
        'LDca5163/LDca5581', 'LDca56d2/LDca5d2b'
    ]
    
    data_dicts = []
    for dataset_dir in sorted(base_path.iterdir()):
        if not dataset_dir.is_dir(): continue
        for f in dataset_dir.rglob('*.npy'):
            if any(anomaly in f.as_posix() for anomaly in anomalies):
                continue
            data_dicts.append({"image": f.as_posix()})
            
    print(f"Total valid 3D volumes loaded: {len(data_dicts)}")

    train_transforms = Compose([
        LoadImaged(keys=["image"], reader="NumpyReader"), 
        RandZoomd(keys=["image"], prob=0.5, min_zoom=0.9, max_zoom=1.1, mode="trilinear", keep_size=False),
        SpatialPadd(keys=["image"], spatial_size=(96, 128, 128), mode="constant", constant_values=0),
        RandSpatialCropd(keys=["image"], roi_size=(96, 128, 128), random_size=False),
        ToTensord(keys=["image"])
    ])

    dataset = Dataset(data=data_dicts, transform=train_transforms)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, drop_last=True)
    return dataloader

# ==========================================
# 3. Robust training loop
# ==========================================
def train_foundation_model():
    os.environ["CUDA_VISIBLE_DEVICES"] = "2"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using GPUs: {os.environ['CUDA_VISIBLE_DEVICES']}")
    
    model = MultiModal_nnUNetV2_MIM(
        in_channels=2, 
        patch_size=(12, 16, 16), 
        mask_ratio=0.5
    )
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    # Pure CNN networks can generally use a slightly larger initial learning rate during pre-training
    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-5)
    
    num_epochs = 200
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    
    scaler = GradScaler('cuda')
    dataloader = get_pretraining_dataloader(batch_size=16)
    
    start_epoch = 0
    best_loss = float('inf')
    
    # [Modify save path] Prevent overwriting the previously trained SwinUNETR weights
    latest_ckpt_path = "nnunet_v2_mim_latest.pth"
    best_ckpt_path = "nnunet_v2_mim_best.pth"
    
    if os.path.exists(latest_ckpt_path):
        print(f"=> Loading checkpoint from '{latest_ckpt_path}'")
        checkpoint = torch.load(latest_ckpt_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict']) 
        start_epoch = checkpoint['epoch']
        best_loss = checkpoint['best_loss']
        print(f"=> Resumed from epoch {start_epoch+1}")
    else:
        print("=> Starting nnU-Net v2 MIM pre-training from scratch")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        epoch_loss = 0.0
        
        current_lr = optimizer.param_groups[0]['lr']
        progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{num_epochs} [LR: {current_lr:.2e}]")
        
        for step, batch_data in progress_bar:
            inputs = batch_data["image"].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            
            with autocast('cuda'):
                loss, _ = model(inputs)
                loss = loss.mean()
            
            scaler.scale(loss).backward()
            
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})
            
        scheduler.step()
            
        avg_loss = epoch_loss / len(dataloader)
        print(f"\n[Epoch {epoch+1}] Average Global Loss: {avg_loss:.4f}\n")
        
        state_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_loss': best_loss
        }
        
        torch.save(state_dict, latest_ckpt_path)
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(state_dict, best_ckpt_path)
            print(f"*** New Best nnU-Net v2 Model Saved! Loss: {best_loss:.4f} ***")

if __name__ == '__main__':
    train_foundation_model()